# 03 · Functions, sweeps and devices / Function နှင့် parameter sweep

**Goal:** write once, bind inputs, reuse blocks, and choose a backend. About 15 minutes.
All required cells use CPU. The optional CUDA cell prints a clear skip when unavailable.

Circuit ကို တစ်ကြိမ်ရေးပြီး parameter များ ပြောင်းလဲစမ်းသပ်ပါ။ GPU မရှိလည်း ဤ lesson ကို လေ့လာနိုင်ပါသည်။


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert tuple(map(int, nexqus.__version__.split('.')[:2])) >= (0, 6), 'Install this SDK with: python -m pip install -e .'
print('NexQus SDK:', nexqus.__version__)


NexQus SDK: 1.0.0rc1


In [2]:
from nexqus import circuit

@circuit(2)
def entangler(q, theta):
    q.ry(theta, 0).cx(0, 1)

theta = Parameter('theta')
template = entangler(theta)
sim = Simulator()
bindings = [{'theta': float(t)} for t in np.linspace(0, np.pi, 9)]
results = sim.run_sweep(template, bindings, shots=0, seed=42, retain_state=True)
measured = np.array([r.expectation('ZI') for r in results])
np.testing.assert_allclose(measured, np.cos([v['theta'] for v in bindings]), atol=1e-12)
print('Parameters:', template.parameters)
print('Expectations:', measured)


Parameters: ('theta',)
Expectations: [ 1.00000000e+00  9.23879533e-01  7.07106781e-01  3.82683432e-01
  1.99673462e-16 -3.82683432e-01 -7.07106781e-01 -9.23879533e-01
 -1.00000000e+00]


In [3]:
prepared = sim.compile(template.bind({'theta': .7}))
a = prepared.run(shots=100, seed=7)
b = prepared.run(shots=100, seed=7)
assert a.counts == b.counts
print('Reusable counts:', a.counts)
print('Gate cache:', sim.cache_info())
larger = Circuit(3).compose(entangler(.7), wires=(2, 0))
assert larger.n_qubits == 3
print('Remapped block:', larger)


Reusable counts: {'00': 90, '11': 10}
Gate cache: {'entries': 11, 'bytes': 896, 'limit_bytes': 67108864, 'hits': 0, 'misses': 11}
Remapped block: Circuit(n_qubits=3, operations=2)


In [4]:
# Optional CUDA check. Tiny circuits are not a GPU speed benchmark.
import importlib.util
gpu_available = False
if importlib.util.find_spec('torch') is not None:
    import torch
    gpu_available = torch.cuda.is_available()
if gpu_available:
    from nexqus.validation import compare_results
    q = algorithms.ghz(3)
    cpu_result = Simulator().run(q, shots=0)
    gpu_result = Simulator(backend='torch', device='cuda:0').run(q, shots=0)
    print('CUDA parity:', compare_results(cpu_result, gpu_result)['check'])
else:
    print('SKIPPED optional CUDA cell: compatible PyTorch/CUDA is unavailable.')


SKIPPED optional CUDA cell: compatible PyTorch/CUDA is unavailable.


The SDK's tested RTX 2070 experiment favored GPU batch 256 for one 10-qubit noisy workload.
That is a measured example, not a universal default. Memory grows with batch size; CPU can favor smaller batches.
Only use GPU through `backend='torch', device='cuda:0'`; NumPy is CPU-only and CUDA never silently falls back.

GPU ပိုမြန်မည်ဟု အမြဲမယူဆပါနှင့်။ Circuit အရွယ်အစားနှင့် batch size ပေါ် မူတည်ပါသည်။


In [5]:
result = Simulator(method='trajectory').run(algorithms.ghz(3),
    noise=NoiseModel(Noise.depolarizing(.02)), trajectories=128, batch_size=32,
    shots=0, seed=42)
assert result.metadata['batch_size'] == 32
print({key: result.metadata[key] for key in ['backend', 'device', 'method', 'batch_size', 'estimated_working_bytes']})


{'backend': 'numpy', 'device': 'cpu', 'method': 'trajectory', 'batch_size': 32, 'estimated_working_bytes': 24576}


**Try it:** stream a large sweep with `sim.iter_sweep(...)` rather than retaining all results.
Keep backend, precision, RNG engine, batch layout and SDK version fixed for reproducibility.
